This is a script that wrangles the different LUNAR functions.
This file was located in the LUNAR directory, thus askdirectory was used to get to remote paths where the mol2 files were stored.

Multiple systems are created with cell_builder.
Systems are automatically built to get as close as possible to 50'000/8 atoms (the systems are replicated later on in LAMMPS).

This is an example for generating the DGEBA/AFD - OPS systems.
For different systems, you need to change the file names and stoechiometry according to the table below.
The ber_pre and ber_post lines can be commented for DGEBA/DDS.

|      System      | nb DGEBA (Mw=344.447) | nb AFD (Mw=248.360) | nb DDS (Mw=248.298) | nb OAPS (Mw=1153.628) | nb OPS (Mw=1033.510) | POSS wt % | epoxy - NH2 stoechimetry |
| :--------------: | :-------------------: | :-----------------: | :-----------------: | :-------------------: | :------------------: | :-------: | :----------------------: |
|    DGEBA/AFD     |          94           |         47          |          -          |           -           |          -           |     -     |     94x2 - 47x4 = 0      |
|    DGEBA/DDS     |          92           |                     |         46          |           -           |          -           |     -     |     92x2 - 46x4 = 0      |
| DGEBA/AFD - OAPS |          48           |         20          |          -          |           1           |          -           | 5.092317  | 48x2 - (20x4 + 1x16) = 0 |
| DGEBA/AFD - OPS  |          42           |         21          |          -          |           -           |          1           | 4.988983  |     42x2 - 21x4 = 0      |

# Select files

In [ ]:
from tkinter.filedialog import askdirectory
from time import time

main_path = askdirectory()+"/"

files = [
         {'name':'AFD',          'ext':'.mol2', 'stoe':21, 'tag':'data1'},
         {'name':'ber_pre',      'ext':'.mol2', 'stoe':0,  'tag':'pre1'},
         {'name':'ber_post',     'ext':'.mol2', 'stoe':0,  'tag':'post1'},
         {'name':'DGEBA',        'ext':'.mol2', 'stoe':42, 'tag':'data2'},
         {'name':'curing1_pre',  'ext':'.mol2', 'stoe':0,  'tag':'pre2'},
         {'name':'curing1_post', 'ext':'.mol2', 'stoe':0,  'tag':'post2'},
         {'name':'curing2_pre',  'ext':'.mol2', 'stoe':0,  'tag':'pre3'},
         {'name':'curing2_post', 'ext':'.mol2', 'stoe':0,  'tag':'post3'},
         {'name':'OPS',          'ext':'.mol2', 'stoe':1,  'tag':'data3'},
         ]

print_options = False
commandline_inputs = []

auto_box = True # triggers the duplication of current stoechiometry to match as closely as possible nb_atoms
nb_atoms = 50000/8 # number of atoms in the auto_box
n_duplicate = 1 # if autobox is False, number of times current stoechiometry is duplicated

repeat_builder = 5 # number of different initial configurations created by cell_builder

# atom_typing

In [ ]:
for i in files:
    topofile = main_path + "0_molecules/" + i['name'] + i['ext']
    bondfile = 'n.u.'
    parent_directory = main_path + "1_atom_typing/"
    chargefile = 'frc_files/Gasteiger_parameters.txt'
    newfile = ':_typed'

    ff_name = 'PCFF'

    reset_charges = False
    delete_atoms = {'method': 'mass', 'criteria': 0.0}
    pdb_file = 'skip'
    include_comments_nta = True

    vdw_radius_scale = 1.1
    boundary = 'p p p'
    bonds_via_distance_override = False

    import src.masses as masses
    mass_map = masses.mass_map
    bondorder = {}
    maxbonded = {}

    from src.atom_typing.main import main
    main(topofile, bondfile, parent_directory, newfile, ff_name, delete_atoms, mass_map, bondorder, maxbonded, boundary, vdw_radius_scale, reset_charges, print_options, commandline_inputs, bonds_via_distance_override, pdb_file, chargefile, include_comments_nta)



Running atom_typing v1.11 / 14 April 2025
Using Python version 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]
Assigning PCFF atom-types
Read in 0_molecules/DDS.mol2 SYBYL MOL2 file
Finding molecules ...
Finding rings ...
Finding fused-ring clusters ...
Finding atoms hybridization ...


Elements found in system:
- C
- H
- N
- O
- S


Total system mass: 248.15400000000005
--------------------------------------------Cluster Analysis-------------------------------------
  molID     Molecule Size          Mass              %Mass           %Size      Molecule Formula
-------------------------------------------------------------------------------------------------
    1              29             248.15            100.00          100.00      C12-H12-N2-O2-S1


By-products criteria: {'method': 'mass', 'criteria': 0.0}
------By Products Tally------
Type                    Count
-----------------------------


----Inputs used for find_rings----
Walked along elements  :  ['C', 'H', 'N', 'O

# all2lmp

In [ ]:
for i in files:
    filename = main_path + "1_atom_typing/" + i['name'] + "_typed"
    topofile = filename + '.data'
    nta_file = filename + '.nta'
    frc_file = 'frc_files/pcff.frc'
    assumed = 'frc_files/general_assumed_equivs.coeffs'
    parent_directory = main_path + "2_all2lmp/"
    newfile = ':_IFF'

    ff_class = '2'
    atom_style = 'full'
    use_auto_equivalence = True
    use_morse_bonds = False
    use_assumed_auto_fill = False
    add2box = 0.0

    reset_molids = True
    if 'data' in i['tag']:
        reset_charges = True
    else:
        reset_charges = False

    # reset_charges = True
    write_txt_comments = True
    write_bond_react = True
    include_type_labels = True
    ignore_missing_parameters = True

    shift = {'x': 0.0, 'y': 0.0, 'z': 0.0}
    rotate = {'x': 0.0, 'y': 0.0, 'z': 0.0}

    print_options = False
    commandline_inputs = []

    from src.all2lmp.main import main
    main(topofile, nta_file, frc_file, assumed, parent_directory, newfile, atom_style, ff_class, use_auto_equivalence, use_assumed_auto_fill, reset_molids, reset_charges, write_txt_comments, write_bond_react, print_options, use_morse_bonds, include_type_labels, add2box, ignore_missing_parameters, shift, rotate, commandline_inputs)



Running all2lmp v1.22 / 14 April 2025
Using Python version 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]
Trying Atom Equivalences if needed
Trying Atom Auto-Equivalences if needed
Trying to find atom charge based on bond-incs
Resetting molecule-ids based on cluster analysis


Using higher version of parameters for #equivalences N a version: 2.10 
Using higher version of parameters for #bond_increments h si version: 2.20 
Using higher version of parameters for #bond_increments c_1 cp version: 2.10 
Using higher version of parameters for #bond_increments si si version: 2.20 
Using higher version of parameters for #bond_increments cp si version: 2.20 
Using higher version of parameters for #bond_increments c si version: 2.20 
Using higher version of parameters for #quartic_bonds c_1 o_1 version: 2.10 
Using higher version of parameters for #quartic_bonds c c version: 2.10 
Using higher version of parameters for #quartic_bonds c o_2 version: 2.20 
Using higher version of parameters 

# bond_react_merge

In [ ]:
files_rxn = { i['tag']  : main_path + "2_all2lmp/" + i['name'] +"_typed_IFF.data" for i in files }
newfile = ':_merged'
parent_directory = main_path + "3_bond_react_merge/"

atom_style = 'full'
generate_map_file = False
write_rxn_mol2files = False
write_moleculefiles = True
write_rxn_datafiles = True

map_near_edge_rxn_charges = False
include_type_labels = True

molecule_file_options = ['']

print_options = False
commandline_inputs = []

from src.bond_react_merge.main import main
main(files_rxn, parent_directory, newfile, atom_style, generate_map_file, write_rxn_mol2files, write_rxn_datafiles, write_moleculefiles, print_options, commandline_inputs, map_near_edge_rxn_charges, molecule_file_options, include_type_labels)



Running bond_react_merge v1.16 / 2 November 2024
Using Python version 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]


---------------------------------------------------------------------------------------------
|                read in files key to value dictionary map to organize files                |
---------------------------------------------------------------------------------------------
|      key to      |                          value to dictionary                           |
|    dictionary    |                                filename                                |
---------------------------------------------------------------------------------------------
|      data1       |                           DDS_typed_IFF.data                           |
|      data2       |                          DGEBA_typed_IFF.data                          |
|       pre2       |                        poly1_pre_typed_IFF.data                        |
|      post2       |         

# cell_builder

In [ ]:
if auto_box:
    unit = 0
    for i in files:
        if i['stoe']: # only counts structures that are in the initial systems (no pre/post templates)
            filename = main_path + "3_bond_react_merge/" + i['name'] + "_typed_IFF_merged.data"
            with open(filename, 'rt') as file_in:
                lines = file_in.readlines()
            for line in lines:
                if " atoms" in line:
                    count = int(line.split(" ")[0].strip()) # get the number of atoms in the molecule
                    break
            unit += i['stoe'] * count # total number of atoms in the unit cell

    nb_unit = nb_atoms / unit
    print("Exact ratio: {}".format(nb_unit))
    nb_unit = round(nb_unit)
    print("Round ratio: {}, nb of atoms: {}".format(nb_unit, nb_unit*unit))


Exact ratio: 2.123683316343867
Round ratio: 2, nb of atoms: 5886


In [ ]:
for i in range(repeat_builder):
    files_cell = { main_path + "3_bond_react_merge/" + i['name'] +"_typed_IFF_merged.data" : i['stoe'] for i in files if i['stoe'] }

    parent_directory = main_path + "4_cell_builder/" + str(i)
    newfile = 'system_cell'
    if auto_box:
        duplicate = nb_unit
    else:
        duplicate = n_duplicate

    domain = 'cubic'
    distance_scale = 1.0
    atom_style = 'full'
    include_type_labels = True
    reset_molids = 'skip'
    unwrap_atoms_via_image_flags = True

    group_monomers_locally = False
    seed = time() # just to get a random initial state
    max_rotations = {'x': 360.0, 'y': 360.0, 'z': 360.0}
    force_field_joining = 'none'

    maxtry = 100
    tolerance = 2.0
    mixing_rule = 'tolerance'
    boundary = 'p p p'

    from src.cell_builder.main import main
    main(files_cell, force_field_joining, duplicate, distance_scale, newfile, atom_style, parent_directory, max_rotations, reset_molids, unwrap_atoms_via_image_flags, include_type_labels, group_monomers_locally, seed, domain, maxtry, tolerance, mixing_rule, boundary, commandline_inputs=commandline_inputs)



Running cell_builder v1.10 / 4 October 2024
Using Python version 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]



----------------------------------------------
              Subcell info
----------------------------------------------
Total number of subcells required:  128
Lattice spacing for subcells     : 17.078888034695737 angstrom


Generating new simulation cell ....
    progress:  5  %,  time:     0.0189 seconds,  density: 0.00169 g/cc  
    progress: 10  %,  time:     0.0327 seconds,  density: 0.00338 g/cc  
    progress: 16  %,  time:     0.0469 seconds,  density: 0.00507 g/cc  
    progress: 21  %,  time:     0.0644 seconds,  density: 0.00676 g/cc  
    progress: 27  %,  time:     0.0765 seconds,  density: 0.00845 g/cc  
    progress: 32  %,  time:     0.0895 seconds,  density: 0.01013 g/cc  
    progress: 38  %,  time:     0.1151 seconds,  density: 0.01248 g/cc  
    progress: 43  %,  time:     0.1381 seconds,  density: 0.01482 g/cc  
    progress: 49  %,  time:     0